### Basic commands dataset creation

In [1]:
import random
import ollama
import sys
sys.path.insert(1, '../../')
from common import get_prompt_template, is_command_classification_correct, to_output_version_1, parse_llm_output_to_list_of_lists, command_to_index
import json
import re
import os

os.environ["OLLAMA_HOST"] = "" # AVS host

In [2]:
dataset = {}

**v2.0** - Introduces five most basic commands:
START, STOP, ENGINE_STOP, ENGINE_START, CRANK_REQUEST

**STOP**

In [ ]:
def get_deepseek_response(text_command, correct_answer):
    print(f"Processing command: {text_command}") #logging
    messages=[
            {"role": "system",  "content": get_prompt_template(command_list=["STOP", "START", "ENGINE_STOP", "ENGINE_START", "CRANK_REQUEST"], prompt_version=2)},
            {"role": "user",  "content": text_command}]
    probability_threshold = 0.80
    client = ollama.Client()
    # we use deepseek-r1-32b here as it gave results of the best accuracy for the subset of commands and the second prompt template
    response = client.chat(model="deepseek-r1:32b", messages=messages)
    if is_command_classification_correct(to_output_version_1(response['message']['content']), correct_answer, probability_threshold)[0] == "CLASSIFICATION_CORRECT":
        response = parse_llm_output_to_list_of_lists(response['message']['content'])
        return str(response)
    else:
        # if the response is not correct, we return an empty string
        return ""


def process_files(chatgpt_input_path: str, deepseek_input_path: str, command_class: str) -> list:

    with open(chatgpt_input_path) as file:
        chatgpt_input = file.read()

    with open(deepseek_input_path) as file:
        deepseek_input = file.read()

    # join chatgpt and deepseek inputs
    # split the lines, remove non-alphanumeric characters, remove trailing whitespaces, and convert to lowercase
    data = chatgpt_input.split("\n") + deepseek_input.split("\n")
    for i in range(len(data)):
        data[i] = re.sub(r'[^a-zA-Z0-9]', ' ', data[i]).rstrip().lower()

    # remove duplicates
    data = set(data)
    # make a dictionary with predicted values
    #TODO: Handling of values is not implemented!
    data = { sample : get_deepseek_response(sample, [command_class, None]) for sample in data }
    return data

stop_commands = process_files("./commands/stop/stop_chatgpt.txt", "./commands/stop/stop_deepseek.txt", "STOP")
dataset["STOP"] = stop_commands


Processing command: end driving immediately
Processing command: stop the car without pause
Processing command: no driving halt
Processing command: stop the car right now
Processing command: stand down halt the car
Processing command: terminate driving now
Processing command: abort driving immediately
Processing command: stop driving  please
Processing command: immediately stop the car
Processing command: please stop the car
Processing command: do not proceed further
Processing command: please bring the vehicle to a halt
Processing command: come to a full stop
Processing command: halt the vehicle
Processing command: halt the vehicle s movement
Processing command: full stop now
Processing command: stop right now
Processing command: no driving stop now
Processing command: stand by stop the car
Processing command: cease the car s motion
Processing command: stop the car as soon as you can
Processing command: cease driving  stop now
Processing command: stop the vehicle immediately
Processing

**START**

In [4]:
start_commands = process_files("./commands/start/start_chatgpt.txt", "./commands/start/start_deepseek.txt", "START")
dataset["START"] = start_commands


Processing command: start moving forward
Processing command: let s move out
Processing command: set out now
Processing command: time to set the wheels in motion
Processing command: let s get on the road
Processing command: start your drive
Processing command: initiate motion
Processing command: begin travel
Processing command: drive away
Processing command: start the travel
Processing command: activate motion
Processing command: move out
Processing command: commence movement
Processing command: engage forward motion
Processing command: let s proceed driving
Processing command: start the advance motion
Processing command: commence the drive
Processing command: move the vehicle
Processing command: let s proceed
Processing command: proceed driving
Processing command: drive it
Processing command: initiate movement
Processing command: begin the drive
Processing command: time to start
Processing command: start the drive
Processing command: begin transit
Processing command: set the car in mot

**ENGINE_STOP**

In [5]:
engine_stop_commands = process_files("./commands/engine_stop/engine_stop_chatgpt.txt", "./commands/engine_stop/engine_stop_deepseek.txt", "ENGINE_STOP")
dataset["ENGINE_STOP"] = engine_stop_commands

Processing command: shut the engine off now
Processing command: shut off the drive
Processing command: shut off the engine immediately
Processing command: disconnect the engine power
Processing command: engine termination
Processing command: turn off the system engine
Processing command: end motor activity
Processing command: deactivate the engine
Processing command: engine stop at once
Processing command: switch off the engine
Processing command: engine halt
Processing command: shut off the engine right away
Processing command: disengage power to the engine
Processing command: turn engine off now
Processing command: end the motor function
Processing command: cut the engine immediately
Processing command: end the engine s operation
Processing command: engine shutdown right now
Processing command: turn off the engine manually
Processing command: stop all engine functions
Processing command: engine deactivate now
Processing command: stop the motor now
Processing command: engine stop now


**ENGINE_START**

In [6]:
engine_start_commands = process_files("./commands/engine_start/engine_start_chatgpt.txt", "./commands/engine_start/engine_start_deepseek.txt", "ENGINE_START")
dataset["ENGINE_START"] = engine_start_commands

Processing command: turn on the motor
Processing command: get the motor started
Processing command: push the engine start button
Processing command: engage the engine
Processing command: hit the start engine button
Processing command: activate the vehicle engine
Processing command: push the button to start the engine
Processing command: hit the ignition
Processing command: fire the engine up
Processing command: switch the engine on
Processing command: start the car engine
Processing command: turn the engine start key
Processing command: boot up the engine
Processing command: get the engine running
Processing command: execute engine start
Processing command: start engine operation
Processing command: start up the engine
Processing command: run the engine
Processing command: power the motor on
Processing command: turn on the racing engine
Processing command: fire up the engine
Processing command: commence start engine operation
Processing command: start the vehicle engine
Processing comm

**CRANK_REQUEST**

In [7]:
crank_request_commands = process_files("./commands/crank_request/crank_request_chatgpt.txt", "./commands/crank_request/crank_request_deepseek.txt", "CRANK_REQUEST")
dataset["CRANK_REQUEST"] = crank_request_commands

Processing command: go for engine cranking
Processing command: activate the auto crank function
Processing command: kickstart the cranking process
Processing command: engine ignition sequence started
Processing command: revolving the crankshaft
Processing command: start the continuous crank
Processing command: initiate the starter motor cranking
Processing command: put the engine in crank mode
Processing command: let s get the engine turning
Processing command: the vehicle is cranking
Processing command: execute engine cranking
Processing command: engage the backup crank
Processing command: let s get cranking
Processing command: run the engine s cranking routine
Processing command: begin the pulsed crank
Processing command: engaging the ignition system
Processing command: engage the high speed crank
Processing command: the vehicle is attempting to crank
Processing command: begin the warm up crank
Processing command: begin the manual override crank
Processing command: firing up the star

Export dataset

In [8]:
with open("../../data/basic_commands_v2.0.json", 'w') as file:
    file.write(json.dumps(dataset, indent=4))

Manually complement the commands for which the prediction failed. Here, some commands have been removed after manual validation.

In [ ]:
all_commands = 0
complemented_manually = 0

with open("../../data/basic_commands_v2.0.json", 'r') as file:
    dataset = json.load(file)

for command_class, commands in dataset.items():
    for command, response in commands.items():
        all_commands += 1
        if response == "":
            complemented_manually += 1
            commands[command] = str([[i, None, 1.0 if command_to_index[command_class] == i else 0.0] for i in range(1, 6)])

print(f"Total number of commands: {all_commands}")
print(f"Number of commands complemented manually: {complemented_manually}")
print(f"Percentage of commands complemented manually: {complemented_manually / all_commands * 100:.2f}%")
# Save the dataset to a file
with open("../../data/basic_commands_v2.0.json", 'w') as file:
    json.dump(dataset, file, indent=4)

Total number of commands: 657
Number of commands complemented manually: 164
Percentage of commands complemented manually: 24.96%


Add some noise - some irrelevant sentences

In [ ]:
with open("../../data/basic_commands_v2.0.json", 'r') as file:
    dataset = json.load(file)


with open("./commands/other/other.txt") as file:
    irrelevant_sentences = file.read()

# split the lines, remove non-alphanumeric characters, remove trailing whitespaces, and convert to lowercase
data = irrelevant_sentences.split("\n")
for i in range(len(data)):
    data[i] = re.sub(r'[^a-zA-Z0-9]', ' ', data[i]).rstrip().lower()

data = { sample : "[[1, None, 0.0], [2, None, 0.0], [3, None, 0.0], [4, None, 0.0], [5, None, 0.0]]" for sample in data }


dataset["OTHER"] = data


with open("../../data/basic_commands_v2.0.json", 'w') as file:
    json.dump(dataset, file, indent=4)

Split the dataset into train and test subsets

In [9]:
with open("../../data/basic_commands_v2.0.json", 'r') as file:
    dataset = json.load(file)

train_set = {}
test_set = {}
for command_class, commands in dataset.items():
    all_commands = list(commands.items())
    random.shuffle(all_commands)

    split_index = int(0.8 * len(all_commands))
    train_commands = all_commands[:split_index]
    test_commands = all_commands[split_index:]

    train_set[command_class] = dict(train_commands)
    test_set[command_class] = dict(test_commands)

with open("../../data/basic_commands_train_v2.0.json", 'w') as file:
    json.dump(train_set, file, indent=4)
with open("../../data/basic_commands_test_v2.0.json", 'w') as file:
    json.dump(test_set, file, indent=4)

In [10]:
# Calculate the percentage of commands in the train and test sets
train_set_percentage = {}
test_set_percentage = {}
for command_class, commands in dataset.items():
    train_set_percentage[command_class] = len(train_set[command_class]) / len(commands) * 100
    test_set_percentage[command_class] = len(test_set[command_class]) / len(commands) * 100
print("Train set percentage:")
for command_class, percentage in train_set_percentage.items():
    print(f"{command_class}: {percentage:.2f}%")
print("Test set percentage:")
for command_class, percentage in test_set_percentage.items():
    print(f"{command_class}: {percentage:.2f}%")

Train set percentage:
STOP: 79.85%
START: 79.65%
ENGINE_STOP: 80.00%
ENGINE_START: 80.00%
CRANK_REQUEST: 80.00%
OTHER: 79.70%
Test set percentage:
STOP: 20.15%
START: 20.35%
ENGINE_STOP: 20.00%
ENGINE_START: 20.00%
CRANK_REQUEST: 20.00%
OTHER: 20.30%
